In [3]:
import pandas as pd

In [4]:
data = pd.read_csv('../data/merged/clean_data.csv') # z notatnika cleaning.ipynb

In [5]:
data.columns

Index(['tmdbId', 'title', 'budget', 'revenue', 'release_date', 'runtime',
       'original_language', 'popularity', 'vote_average', 'vote_count',
       'origin_countries', 'spoken_languages', 'year', 'keywords', 'genres',
       'director_id', 'director_name', 'director_popularity',
       'director_gender', 'writer_id', 'writer_name', 'writer_popularity',
       'writer_gender', 'actor4_id', 'actor4_name', 'actor4_popularity',
       'actor4_gender', 'actor2_id', 'actor2_name', 'actor2_popularity',
       'actor2_gender', 'actor1_id', 'actor1_name', 'actor1_popularity',
       'actor1_gender', 'actor5_id', 'actor5_name', 'actor5_popularity',
       'actor5_gender', 'actor3_id', 'actor3_name', 'actor3_popularity',
       'actor3_gender', 'budget_adjusted', 'revenue_adjusted', 'quarter',
       'epoka', 'main_genre', 'one_or_more_origin', 'origin_in_US',
       'origin_in_top_5'],
      dtype='object')

## Usuwanie braków i złych wartości zmiennych

In [6]:
data = data[['tmdbId', 'title','release_date', 'runtime',
       'original_language', 'vote_average', 'vote_count',
       'origin_countries', 'spoken_languages', 'year', 'keywords', 'genres',
       'director_id', 'director_name', 'director_gender',
       'writer_id', 'writer_name', 'writer_gender',
       'actor4_id', 'actor4_name', 'actor4_gender',
       'actor2_id', 'actor2_name', 'actor2_gender',
       'actor1_id', 'actor1_name', 'actor1_gender',
       'actor5_id', 'actor5_name', 'actor5_gender',
       'actor3_id', 'actor3_name', 'actor3_gender',
       'budget_adjusted', 'revenue_adjusted', 'quarter', 'main_genre']]
data.shape

(19120, 37)

In [7]:
data = data[data["budget_adjusted"] > 0]
data.shape

(11583, 37)

In [8]:
data = data[data["year"] > 1979]

In [9]:
data.shape

(10591, 37)

In [10]:
data = data[data["runtime"] > 60]
data.shape

(10579, 37)

In [11]:
data.isna().sum()

tmdbId                 0
title                  0
release_date           0
runtime                0
original_language      0
vote_average           0
vote_count             0
origin_countries       0
spoken_languages     130
year                   0
keywords             948
genres                13
director_id            9
director_name          9
director_gender        9
writer_id            199
writer_name          199
writer_gender        199
actor4_id             96
actor4_name           96
actor4_gender         96
actor2_id             18
actor2_name           18
actor2_gender         18
actor1_id              8
actor1_name            8
actor1_gender          8
actor5_id            371
actor5_name          371
actor5_gender        371
actor3_id             43
actor3_name           43
actor3_gender         43
budget_adjusted        0
revenue_adjusted       0
quarter                0
main_genre            13
dtype: int64

In [12]:
data = data.dropna()
data.shape

(9187, 37)

Po usunięciu wszystkich brakujących danych pozostało ponad 9tys danych. Większość braków była w starych filmach przed rokiem 1980, tak samo większość braków w budżecie pochodziła z tamtego okresu oraz większość filmów krótszych niż 1 godzina.

## Język oryginalny

In [13]:
# Zostawiamy 20 najczęstszych języków + kategoria other na wszystkie inne
data["original_language"].value_counts()

original_language
en       7233
fr        328
hi        260
other     249
es        163
ja        123
ru        116
ko         97
zh         91
it         81
ar         74
ta         71
de         59
cn         57
te         43
no         38
pt         29
sv         24
ml         19
tr         18
pl         14
Name: count, dtype: int64

## Kraj pochodzenia

In [14]:
# Zostawiamy 20 najczęstszych krajów + kategoria other na wszystkie inne
# Bierzemy pod uwagę tylko pierwszy kraj, jeśli pojawa się więcej na liście

data['origin_countries'] = data['origin_countries'].str.replace(r"[\[\]']", "", regex=True).str.strip()

wszystkie_kraje = data['origin_countries'].str.split(', ').explode()
top_20_kraje = wszystkie_kraje.value_counts().head(20).index.tolist()

def kategoryzuj_kraj(row):
    kraje = row.split(', ')
    glowny_kraj = kraje[0] # Bierzemy pierwszy wymieniony kraj jako dominujący
    
    if glowny_kraj == 'UnitedStatesofAmerica':
        return 'USA'
    elif glowny_kraj in top_20_kraje:
        return glowny_kraj
    else:
        return 'Other'

data['main_country'] = data['origin_countries'].apply(kategoryzuj_kraj)


print(data['main_country'].value_counts())

main_country
USA              5048
UnitedKingdom     658
Other             640
France            501
India             430
Canada            348
Germany           273
Australia         170
China             155
Japan             151
Belgium           121
Spain             118
Russia            112
Italy             111
SouthKorea         99
HongKong           75
Denmark            52
Ireland            47
Mexico             37
Sweden             22
Netherlands        19
Name: count, dtype: int64


## Keywords

In [15]:
all_keywords = data['keywords'].str.split(',').explode().str.strip()

all_keywords = all_keywords[all_keywords != ""]

top_100_keywords = all_keywords.value_counts().head(100)

print(top_100_keywords.to_string())


keywords
original                  860
action                    574
comedy                    493
horror                    339
drama                     328
family                    317
sequel                    316
based on true story       308
based on a book           298
based on novel or book    293
murder                    241
remake                    239
woman director            236
animation                 224
mentor                    223
romantic                  213
predictable               196
supernatural              193
independent film          191
sports                    189
violence                  188
weird                     183
friendship                181
coming of age             179
chick flick               177
adapted from:book         176
chase                     175
relationships             173
visually appealing        172
good action               172
love                      169
suspense                  169
girlie movie              167
t

In [16]:
print(data["main_genre"].value_counts())

main_genre
comedy         2181
action         1447
horror          758
romance         695
thriller        585
crime           542
drama           541
adventure       517
animation       332
family          277
history         272
sci-fi          265
fantasy         258
mystery         158
musical         136
war             130
western          51
documentary      42
Name: count, dtype: int64


In [17]:
wybrane_keywords = ['sequel', 'prequel', 'based', 'murder', 'remake', 'love', 'supernatural', 'violence',
                    'sport', 'friendship', 'weird', 'teen',  'funny', 'alien', 'superhero', 'magic',
                    'chase', 'heist', 'book', 'revenge', 'town', 'city', 'village', 'biography',
                    'police', 'gangster', 'war', 'christmas', 'politics', 'holiday', 'death', 
                    'suspense', 'melancholic', 'drugs', 'alcohol', 'future', 'detective', 'time travel',
                    'true story', 'spy', 'dark']

len(wybrane_keywords)

41

In [18]:
wzorzec = '|'.join(wybrane_keywords)

# Sprawdzamy ile filmów mieliśmy na początku
print(f"Filmy przed filtrowaniem: {len(data)}")

data[data['keywords'].str.contains(wzorzec, case=False, na=False, regex=True)].copy()

Filmy przed filtrowaniem: 9187


,tmdbId,title,release_date,runtime,original_language,vote_average,vote_count,origin_countries,spoken_languages,year,...,actor5_name,actor5_gender,actor3_id,actor3_name,actor3_gender,budget_adjusted,revenue_adjusted,quarter,main_genre,main_country
1,6.0,Judgment Night,1993-10-15,109,en,6.469,368,UnitedStatesofAmerica,English,1993,...,Cuba Gooding Jr.,2.0,12799.0,Jeremy Piven,2.0,4.678756e+07,2.704085e+07,4,action,USA
6,16.0,Dancer in the Dark,2000-09-01,140,en,7.850,1961,"Denmark, Finland, France, Germany, Iceland, Ne...",English,2000,...,Catherine Deneuve,1.0,52.0,David Morse,2.0,2.336985e+07,7.489784e+07,3,crime,Denmark
9,20.0,My Life Without Me,2003-03-07,106,en,6.079,496,"Canada, Spain",English,2003,...,Amanda Plummer,1.0,100.0,Scott Speedman,2.0,4.374226e+06,2.152119e+07,1,romance,Canada
11,24.0,Kill Bill: Vol. 1,2003-10-10,111,en,8.000,18683,UnitedStatesofAmerica,"English, 日本語, Français",2003,...,Vivica A. Fox,1.0,140.0,Lucy Liu,1.0,5.249071e+07,3.165296e+08,4,action,USA
12,25.0,Jarhead,2005-11-04,123,en,6.680,3128,"UnitedKingdom, UnitedStatesofAmerica","English, Español, العربية, Latin",2005,...,Peter Sarsgaard,2.0,134.0,Jamie Foxx,2.0,1.186887e+08,1.600255e+08,4,war,UnitedKingdom
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19090,1500536.0,Dead to Rights,2025-07-19,137,zh,8.000,73,China,"普通话, 日本語",2025,...,Wang Xiao,2.0,1800556.0,Zhou You,2.0,2.000000e+07,4.227859e+08,3,war,China
19096,1511417.0,Bāhubali: The Epic,2025-10-29,224,te,5.985,34,India,"?????, हिन्दी, , தமிழ், తెలుగు",2025,...,Rana Daggubati,2.0,141701.0,Ramya Krishnan,1.0,6.500000e+07,3.500000e+06,4,action,India
19099,1518784.0,Witnessed,2023-05-06,88,en,10.000,2,,English,2023,...,Marcus Hopkins-Turner,2.0,2283871.0,Lau'rie Roach,0.0,1.056583e+04,5.599891e+04,2,action,Other
19106,1537715.0,Banduan,2025-11-06,127,other,6.500,2,Malaysia,Bahasa melayu,2025,...,Abi Madyan,0.0,557749.0,Aaron Aziz,2.0,1.600000e+06,1.472934e+06,4,action,Other


## Ludzie

In [19]:
data.drop_duplicates(subset='tmdbId', keep='first', inplace=True)
data.shape

(9186, 38)

Średnia liczba filmów reżysera, scenarzysty i aktorów

In [20]:
df = data.copy()

director_counts = df['director_id'].value_counts()
df['director_movie_count'] = df['director_id'].map(director_counts)

writer_counts = df['writer_id'].value_counts()
df['writer_movie_count'] = df['writer_id'].map(writer_counts)

for i in range(1, 6):
    counts = df[f'actor{i}_id'].value_counts()
    df[f'actor{i}_movie_count'] = df[f'actor{i}_id'].map(counts)

df['actors_avg_movie_count'] = df[[f'actor{i}_movie_count' for i in range(1,6)]].mean(axis=1)
df.drop(columns=[f'actor{i}_movie_count' for i in range(1,6)], inplace=True)

df

,tmdbId,title,release_date,runtime,original_language,vote_average,vote_count,origin_countries,spoken_languages,year,...,actor3_name,actor3_gender,budget_adjusted,revenue_adjusted,quarter,main_genre,main_country,director_movie_count,writer_movie_count,actors_avg_movie_count
0,5.0,Four Rooms,1995-12-09,98,en,5.892,2810,UnitedStatesofAmerica,English,1995,...,Jennifer Beals,1.0,8.449948e+06,8.993604e+06,4,comedy,USA,15,13,8.2
1,6.0,Judgment Night,1993-10-15,109,en,6.469,368,UnitedStatesofAmerica,English,1993,...,Jeremy Piven,2.0,4.678756e+07,2.704085e+07,4,action,USA,9,9,3.6
3,12.0,Finding Nemo,2003-05-30,100,en,7.817,20364,UnitedStatesofAmerica,English,2003,...,Geoffrey Rush,2.0,1.644709e+08,1.645296e+09,2,animation,USA,9,7,12.4
4,13.0,Forrest Gump,1994-06-23,142,en,8.500,29387,UnitedStatesofAmerica,English,1994,...,Sally Field,1.0,1.194795e+08,1.471527e+09,2,comedy,USA,20,13,21.8
5,14.0,American Beauty,1999-09-15,122,en,8.000,12880,UnitedStatesofAmerica,English,1999,...,Annette Bening,1.0,2.898646e+07,6.885186e+08,3,romance,USA,9,1,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19090,1500536.0,Dead to Rights,2025-07-19,137,zh,8.000,73,China,"普通话, 日本語",2025,...,Zhou You,2.0,2.000000e+07,4.227859e+08,3,war,China,1,1,1.0
19096,1511417.0,Bāhubali: The Epic,2025-10-29,224,te,5.985,34,India,"?????, हिन्दी, , தமிழ், తెలుగు",2025,...,Ramya Krishnan,1.0,6.500000e+07,3.500000e+06,4,action,India,6,1,2.6
19099,1518784.0,Witnessed,2023-05-06,88,en,10.000,2,,English,2023,...,Lau'rie Roach,0.0,1.056583e+04,5.599891e+04,2,action,Other,1,1,1.0
19106,1537715.0,Banduan,2025-11-06,127,other,6.500,2,Malaysia,Bahasa melayu,2025,...,Aaron Aziz,2.0,1.600000e+06,1.472934e+06,4,action,Other,3,1,1.2


Średni dochód z filmów reżysera, scenarzysty i aktorów + maksymalny średni dochód aktorów

In [21]:
df['writer_avg_revenue'] = df.groupby('writer_id')['revenue_adjusted'].transform('mean')
df['writer_max_revenue'] = df.groupby('writer_id')['revenue_adjusted'].transform('max')

df['director_avg_revenue'] = df.groupby('director_id')['revenue_adjusted'].transform('mean')
df['director_max_revenue'] = df.groupby('director_id')['revenue_adjusted'].transform('max')

for i in range(1, 6):
    df[f'actor{i}_avg_revenue'] = df.groupby(f'actor{i}_id')['revenue_adjusted'].transform('mean')
    df[f'actor{i}_max_revenue'] = df.groupby(f'actor{i}_id')['revenue_adjusted'].transform('max')

df['actors_avg_revenue'] = df[[f'actor{i}_avg_revenue' for i in range(1,6)]].mean(axis=1)
df['actors_max_revenue'] = df[[f'actor{i}_max_revenue' for i in range(1,6)]].mean(axis=1)

df

,tmdbId,title,release_date,runtime,original_language,vote_average,vote_count,origin_countries,spoken_languages,year,...,actor2_avg_revenue,actor2_max_revenue,actor3_avg_revenue,actor3_max_revenue,actor4_avg_revenue,actor4_max_revenue,actor5_avg_revenue,actor5_max_revenue,actors_avg_revenue,actors_max_revenue
0,5.0,Four Rooms,1995-12-09,98,en,5.892,2810,UnitedStatesofAmerica,English,1995,...,2.732886e+07,9.082649e+07,1.979667e+07,2.889085e+07,1.979667e+07,2.889085e+07,1.078966e+08,3.039070e+08,3.948193e+07,1.086683e+08
1,6.0,Judgment Night,1993-10-15,109,en,6.469,368,UnitedStatesofAmerica,English,1993,...,1.732123e+07,2.704085e+07,4.697796e+07,6.691507e+07,8.920569e+07,1.554244e+08,1.758768e+08,6.302019e+08,7.128450e+07,1.813246e+08
3,12.0,Finding Nemo,2003-05-30,100,en,7.817,20364,UnitedStatesofAmerica,English,2003,...,1.168164e+08,1.645296e+09,5.784439e+08,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,8.257942e+08,1.645296e+09
4,13.0,Forrest Gump,1994-06-23,142,en,8.500,29387,UnitedStatesofAmerica,English,1994,...,4.328260e+08,1.575811e+09,4.190304e+08,1.471527e+09,4.124310e+08,1.471527e+09,2.926597e+08,1.471527e+09,3.865216e+08,1.513240e+09
5,14.0,American Beauty,1999-09-15,122,en,8.000,12880,UnitedStatesofAmerica,English,1999,...,1.671920e+08,7.174764e+08,8.403468e+07,6.885186e+08,1.374697e+08,6.885186e+08,3.446150e+08,6.885186e+08,1.829980e+08,7.001017e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19090,1500536.0,Dead to Rights,2025-07-19,137,zh,8.000,73,China,"普通话, 日本語",2025,...,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08
19096,1511417.0,Bāhubali: The Epic,2025-10-29,224,te,5.985,34,India,"?????, हिन्दी, , தமிழ், తెలుగు",2025,...,1.294343e+07,2.100897e+07,3.500000e+06,3.500000e+06,3.500000e+06,3.500000e+06,1.860871e+08,3.686741e+08,6.572882e+07,1.530714e+08
19099,1518784.0,Witnessed,2023-05-06,88,en,10.000,2,,English,2023,...,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04
19106,1537715.0,Banduan,2025-11-06,127,other,6.500,2,Malaysia,Bahasa melayu,2025,...,9.578993e+05,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.369927e+06,1.472934e+06


Ilość aktorek w filmie

In [22]:
df['female_actors'] = (
        df[[f'actor{i}_gender' for i in range(1,6)]] == 1
).sum(axis=1)
df

,tmdbId,title,release_date,runtime,original_language,vote_average,vote_count,origin_countries,spoken_languages,year,...,actor2_max_revenue,actor3_avg_revenue,actor3_max_revenue,actor4_avg_revenue,actor4_max_revenue,actor5_avg_revenue,actor5_max_revenue,actors_avg_revenue,actors_max_revenue,female_actors
0,5.0,Four Rooms,1995-12-09,98,en,5.892,2810,UnitedStatesofAmerica,English,1995,...,9.082649e+07,1.979667e+07,2.889085e+07,1.979667e+07,2.889085e+07,1.078966e+08,3.039070e+08,3.948193e+07,1.086683e+08,3
1,6.0,Judgment Night,1993-10-15,109,en,6.469,368,UnitedStatesofAmerica,English,1993,...,2.704085e+07,4.697796e+07,6.691507e+07,8.920569e+07,1.554244e+08,1.758768e+08,6.302019e+08,7.128450e+07,1.813246e+08,0
3,12.0,Finding Nemo,2003-05-30,100,en,7.817,20364,UnitedStatesofAmerica,English,2003,...,1.645296e+09,5.784439e+08,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,8.257942e+08,1.645296e+09,0
4,13.0,Forrest Gump,1994-06-23,142,en,8.500,29387,UnitedStatesofAmerica,English,1994,...,1.575811e+09,4.190304e+08,1.471527e+09,4.124310e+08,1.471527e+09,2.926597e+08,1.471527e+09,3.865216e+08,1.513240e+09,3
5,14.0,American Beauty,1999-09-15,122,en,8.000,12880,UnitedStatesofAmerica,English,1999,...,7.174764e+08,8.403468e+07,6.885186e+08,1.374697e+08,6.885186e+08,3.446150e+08,6.885186e+08,1.829980e+08,7.001017e+08,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19090,1500536.0,Dead to Rights,2025-07-19,137,zh,8.000,73,China,"普通话, 日本語",2025,...,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,1
19096,1511417.0,Bāhubali: The Epic,2025-10-29,224,te,5.985,34,India,"?????, हिन्दी, , தமிழ், తెలుగు",2025,...,2.100897e+07,3.500000e+06,3.500000e+06,3.500000e+06,3.500000e+06,1.860871e+08,3.686741e+08,6.572882e+07,1.530714e+08,2
19099,1518784.0,Witnessed,2023-05-06,88,en,10.000,2,,English,2023,...,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,1
19106,1537715.0,Banduan,2025-11-06,127,other,6.500,2,Malaysia,Bahasa melayu,2025,...,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.369927e+06,1.472934e+06,0


Najpopularniejsi ludzie kina

In [23]:
top_actors = pd.read_csv('../data/top_people/top100_actors.csv')['Name'].to_list()
top_actors_today = pd.read_csv('../data/top_people/top100_actors_today.csv')['Name'].to_list()
top_actors_all = set(top_actors + top_actors_today)
print(len(top_actors_all))

top_directos = pd.read_csv('../data/top_people/top_directors.csv')['Name'].to_list()
print(len(top_directos))
top_writers = pd.read_csv('../data/top_people/top_writers.csv')['Name'].to_list()
print(len(top_writers))
top_writers

185
50
20


['Aaron Sorkin',
 'Quentin Tarantino',
 'Woody Allen',
 'Billy Wilder',
 'Joel Coen',
 'Ethan Coen',
 'Oliver Stone',
 'William Goldman',
 'Steven Zaillian',
 'James Cameron',
 'Mel Brooks',
 'Paul Schrader',
 'Ingmar Bergman',
 'Lawrence Kasdan',
 'Eric Roth',
 'Alan Ball',
 'Paul Thomas Anderson',
 'Walter Hill',
 'John Hughes',
 'David Lynch']

In [24]:
df['top_actors_number'] = (
        df[[f'actor{i}_name' for i in range(1,6)]]
).isin(top_actors_all).sum(axis=1)

df['top_director_number'] = df['director_name'].isin(top_directos).astype(int)
df['top_writer_number'] = df['writer_name'].isin(top_writers).astype(int)

df['top_actors'] = (
        df[[f'actor{i}_name' for i in range(1,6)]]
).isin(top_actors_all).any(axis=1)

df['top_director'] = df['director_name'].isin(top_directos)
df['top_writer'] = df['writer_name'].isin(top_writers)

df['top_people_number'] = df['top_actors_number'] + df['top_director_number'] + df['top_writer_number']
df['top_people'] = df['top_actors'] | df['top_director'] | df['top_writer']

df.drop(columns=['top_actors_number', 'top_director_number', 'top_writer_number', 'top_actors', 'top_director', 'top_writer'], inplace=True)
df

,tmdbId,title,release_date,runtime,original_language,vote_average,vote_count,origin_countries,spoken_languages,year,...,actor3_max_revenue,actor4_avg_revenue,actor4_max_revenue,actor5_avg_revenue,actor5_max_revenue,actors_avg_revenue,actors_max_revenue,female_actors,top_people_number,top_people
0,5.0,Four Rooms,1995-12-09,98,en,5.892,2810,UnitedStatesofAmerica,English,1995,...,2.889085e+07,1.979667e+07,2.889085e+07,1.078966e+08,3.039070e+08,3.948193e+07,1.086683e+08,3,2,True
1,6.0,Judgment Night,1993-10-15,109,en,6.469,368,UnitedStatesofAmerica,English,1993,...,6.691507e+07,8.920569e+07,1.554244e+08,1.758768e+08,6.302019e+08,7.128450e+07,1.813246e+08,0,0,False
3,12.0,Finding Nemo,2003-05-30,100,en,7.817,20364,UnitedStatesofAmerica,English,2003,...,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,8.257942e+08,1.645296e+09,0,4,True
4,13.0,Forrest Gump,1994-06-23,142,en,8.500,29387,UnitedStatesofAmerica,English,1994,...,1.471527e+09,4.124310e+08,1.471527e+09,2.926597e+08,1.471527e+09,3.865216e+08,1.513240e+09,3,4,True
5,14.0,American Beauty,1999-09-15,122,en,8.000,12880,UnitedStatesofAmerica,English,1999,...,6.885186e+08,1.374697e+08,6.885186e+08,3.446150e+08,6.885186e+08,1.829980e+08,7.001017e+08,3,6,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19090,1500536.0,Dead to Rights,2025-07-19,137,zh,8.000,73,China,"普通话, 日本語",2025,...,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,1,0,False
19096,1511417.0,Bāhubali: The Epic,2025-10-29,224,te,5.985,34,India,"?????, हिन्दी, , தமிழ், తెలుగు",2025,...,3.500000e+06,3.500000e+06,3.500000e+06,1.860871e+08,3.686741e+08,6.572882e+07,1.530714e+08,2,0,False
19099,1518784.0,Witnessed,2023-05-06,88,en,10.000,2,,English,2023,...,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,1,0,False
19106,1537715.0,Banduan,2025-11-06,127,other,6.500,2,Malaysia,Bahasa melayu,2025,...,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.369927e+06,1.472934e+06,0,0,False


In [25]:
data = df.copy()

## Zapis do ramki już bez braków danych, odpowiednio przefiltrowanych itp

In [26]:
data.to_csv("../data/merged/NEW_clean_data.csv", index = False)

In [27]:
data

,tmdbId,title,release_date,runtime,original_language,vote_average,vote_count,origin_countries,spoken_languages,year,...,actor3_max_revenue,actor4_avg_revenue,actor4_max_revenue,actor5_avg_revenue,actor5_max_revenue,actors_avg_revenue,actors_max_revenue,female_actors,top_people_number,top_people
0,5.0,Four Rooms,1995-12-09,98,en,5.892,2810,UnitedStatesofAmerica,English,1995,...,2.889085e+07,1.979667e+07,2.889085e+07,1.078966e+08,3.039070e+08,3.948193e+07,1.086683e+08,3,2,True
1,6.0,Judgment Night,1993-10-15,109,en,6.469,368,UnitedStatesofAmerica,English,1993,...,6.691507e+07,8.920569e+07,1.554244e+08,1.758768e+08,6.302019e+08,7.128450e+07,1.813246e+08,0,0,False
3,12.0,Finding Nemo,2003-05-30,100,en,7.817,20364,UnitedStatesofAmerica,English,2003,...,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,1.645296e+09,8.257942e+08,1.645296e+09,0,4,True
4,13.0,Forrest Gump,1994-06-23,142,en,8.500,29387,UnitedStatesofAmerica,English,1994,...,1.471527e+09,4.124310e+08,1.471527e+09,2.926597e+08,1.471527e+09,3.865216e+08,1.513240e+09,3,4,True
5,14.0,American Beauty,1999-09-15,122,en,8.000,12880,UnitedStatesofAmerica,English,1999,...,6.885186e+08,1.374697e+08,6.885186e+08,3.446150e+08,6.885186e+08,1.829980e+08,7.001017e+08,3,6,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19090,1500536.0,Dead to Rights,2025-07-19,137,zh,8.000,73,China,"普通话, 日本語",2025,...,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,4.227859e+08,1,0,False
19096,1511417.0,Bāhubali: The Epic,2025-10-29,224,te,5.985,34,India,"?????, हिन्दी, , தமிழ், తెలుగు",2025,...,3.500000e+06,3.500000e+06,3.500000e+06,1.860871e+08,3.686741e+08,6.572882e+07,1.530714e+08,2,0,False
19099,1518784.0,Witnessed,2023-05-06,88,en,10.000,2,,English,2023,...,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,5.599891e+04,1,0,False
19106,1537715.0,Banduan,2025-11-06,127,other,6.500,2,Malaysia,Bahasa melayu,2025,...,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.472934e+06,1.369927e+06,1.472934e+06,0,0,False
